In [0]:
%run ../utils

# ***Bronze load***

### Check if the file are available

In [0]:
# Check path
display(dbutils.fs.ls(workspace_link + "/data/bronze/full")) # paths & name table require

In [0]:
# Take the last full and compare if the file is up to date
files = [
    f for f in dbutils.fs.ls(workspace_link + "/data/bronze/full")
    if f.name.endswith(".json")
]

latest_file = max(
    files,
    key=lambda x: x.modificationTime
)

print(latest_file.path)

In [0]:
df = (spark.read.option("multiline", "true").json(latest_file.path))

In [0]:
# Check the schema for transform the data
df.printSchema()

In [0]:
# Add the current timestamp when he load + source fil to check if the data is up to date
bronze_df = (
    df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit(latest_file.name))
)

In [0]:
bronze_df.write.mode("overwrite").format("delta").saveAsTable("nasa_meteor.bronze_full")

In [0]:
bronze_full = spark.table("nasa_meteor.bronze_full")
bronze_df.display()